# Predicting Student Health Risk - EDA

This notebook is the public exploratory pass for Kaggle Playground Series S6E7. The goal is not to over-model yet; it is to make the dataset legible, identify the target imbalance, inspect missingness, and turn those observations into a clean modeling checklist.

**Working questions**

- What is the target and how imbalanced is it?
- Which fields are numeric versus categorical?
- Is missingness large enough to become a signal rather than simple cleanup?
- Do train and test distributions look broadly aligned?
- What should the first baseline optimize and diagnose?

## 1. Setup

The data path resolver supports both Kaggle execution and local development. On Kaggle, competition data can mount under different folder names, so the notebook scans `/kaggle/input` when the common paths are absent.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42

candidate_dirs = [
    Path('/kaggle/input/playground-series-s6e7'),
    Path('/kaggle/input/predicting-student-health-risk'),
    Path('../data'),
    Path('data'),
]
DATA_DIR = next(
    (path for path in candidate_dirs if (path / 'train.csv').exists()),
    None,
)
if DATA_DIR is None and Path('/kaggle/input').exists():
    for train_path in Path('/kaggle/input').glob('**/train.csv'):
        parent = train_path.parent
        if (parent / 'test.csv').exists() and (parent / 'sample_submission.csv').exists():
            DATA_DIR = parent
            break
if DATA_DIR is None:
    raise FileNotFoundError('Could not locate train/test/sample_submission CSV files.')

sns.set_theme(style='whitegrid', palette='viridis')
pd.set_option('display.max_columns', 120)
print(f'Using data directory: {DATA_DIR}')

## 2. Load The Competition Files

The competition follows the standard Playground shape: `train.csv`, `test.csv`, and `sample_submission.csv`. The only column present in train but absent from test is treated as the target.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

target_candidates = [col for col in train.columns if col not in test.columns]
target = target_candidates[0]
id_col = sample_submission.columns[0]

summary = pd.DataFrame(
    {
        'rows': [len(train), len(test), len(sample_submission)],
        'columns': [train.shape[1], test.shape[1], sample_submission.shape[1]],
    },
    index=['train', 'test', 'sample_submission'],
)
display(summary)
print(f'Target column: {target}')
display(train.head())
display(sample_submission.head())

## 3. Target Balance

`health_condition` is highly imbalanced. Plain accuracy will be dominated by the majority `at-risk` class, so every serious model should also report class-balanced metrics such as macro F1, balanced accuracy, and per-class recall.

In [ ]:
target_counts = train[target].value_counts(dropna=False)
target_summary = pd.DataFrame(
    {
        'count': target_counts,
        'share': target_counts / len(train),
    }
)
display(target_summary)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(
    x=target_summary.index,
    y=target_summary['share'],
    hue=target_summary.index,
    palette='viridis',
    legend=False,
    ax=ax,
)
ax.set_title('Target class share')
ax.set_xlabel('health_condition')
ax.set_ylabel('Share of training rows')
ax.bar_label(ax.containers[0], fmt='%.1%%')
plt.tight_layout()
plt.show()

## 4. Feature Groups

The dataset mixes continuous lifestyle/health measurements with low-cardinality categorical fields. This is a good fit for tree-based tabular models, especially CatBoost/LightGBM after the first sklearn baseline.

In [ ]:
feature_cols = [col for col in test.columns if col in train.columns and col != id_col]
num_cols = train[feature_cols].select_dtypes(include=np.number).columns.tolist()
cat_cols = [col for col in feature_cols if col not in num_cols]

print(f'ID column: {id_col}')
print(f'Numeric features ({len(num_cols)}): {num_cols}')
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')

display(train[num_cols].describe().T)
cat_summary = pd.DataFrame(
    {
        'train_unique': train[cat_cols].nunique(dropna=False),
        'test_unique': test[cat_cols].nunique(dropna=False),
        'train_missing': train[cat_cols].isna().sum(),
        'test_missing': test[cat_cols].isna().sum(),
    }
)
display(cat_summary)

## 5. Missingness Profile

There are many missing values. For the first baseline, this should be handled explicitly with imputation plus missingness indicators. Later experiments should test whether missingness patterns correlate with `health_condition`.

In [ ]:
missing = pd.DataFrame(
    {
        'train_missing': train[feature_cols].isna().sum(),
        'train_missing_share': train[feature_cols].isna().mean(),
        'test_missing': test[feature_cols].isna().sum(),
        'test_missing_share': test[feature_cols].isna().mean(),
    }
).sort_values('train_missing_share', ascending=False)

display(missing[missing[['train_missing', 'test_missing']].sum(axis=1) > 0])

row_missing = pd.DataFrame(
    {
        'train_missing_features': train[feature_cols].isna().sum(axis=1),
        'target': train[target],
    }
)
display(row_missing.groupby('target')['train_missing_features'].describe())

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=missing.reset_index().rename(columns={'index': 'feature'}),
    x='train_missing_share',
    y='feature',
    color=sns.color_palette('viridis', 5)[2],
    ax=ax,
)
ax.set_title('Missing-value share by feature')
ax.set_xlabel('Train missing share')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 6. Train/Test Drift Quick Screen

This is a lightweight first check. Numeric features compare mean and standard deviation shifts. Categorical features compare the most common train/test values. Larger drift deserves fold-aware validation and cautious leaderboard interpretation.

In [ ]:
num_drift = pd.DataFrame(index=num_cols)
num_drift['train_mean'] = train[num_cols].mean()
num_drift['test_mean'] = test[num_cols].mean()
num_drift['mean_delta'] = num_drift['test_mean'] - num_drift['train_mean']
num_drift['train_std'] = train[num_cols].std()
num_drift['test_std'] = test[num_cols].std()
num_drift['std_delta'] = num_drift['test_std'] - num_drift['train_std']
display(num_drift.sort_values('mean_delta', key=lambda s: s.abs(), ascending=False))

cat_rows = []
for col in cat_cols:
    train_top = train[col].value_counts(dropna=False, normalize=True).head(3)
    test_top = test[col].value_counts(dropna=False, normalize=True).head(3)
    cat_rows.append(
        {
            'feature': col,
            'train_top_values': ', '.join(f'{idx}: {val:.1%}' for idx, val in train_top.items()),
            'test_top_values': ', '.join(f'{idx}: {val:.1%}' for idx, val in test_top.items()),
        }
    )
display(pd.DataFrame(cat_rows))

## 7. Modeling Implications

**Immediate choices**

- Use stratified cross-validation because the target is very imbalanced.
- Report macro F1, balanced accuracy, and per-class recall alongside any official metric.
- Add missing-value indicators to the baseline preprocessing.
- Prefer tree-based models for the first serious pass: HistGradientBoosting, LightGBM, XGBoost, and CatBoost.
- Validate feature families around sleep/stress/activity interactions instead of adding a large untracked feature pile.

**Next notebook:** `02_baseline_modeling.ipynb` creates a fold-safe sklearn baseline and writes a first `submission.csv` artifact.